# A2.1 · Run detumble sim (practice)

**Phase A Concept and Technology Development › Attitude Control (ADCS) › A2 B-dot study + sim** · PM card TC-ADCS-A2

> 💡 This is much easier after you've followed **A2 B-dot study + sim**.

This time you write the B-dot controller **yourself**. The simulator is ready for you.

## Setup (just run this)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Spacecraft: FASTSAT (B-dot paper, section V)
J = np.diag([15.9, 16.9, 11.7])   # moment of inertia [kg·m²]
J_INV = np.linalg.inv(J)
M_MAX = 5.0                        # max magnetorquer dipole moment [A·m²]

# Orbit: 650 km altitude, 72° inclination
MU_E = 3.986e14                    # Earth's gravitational parameter [m³/s²]
R = 6371e3 + 650e3                 # orbit radius [m]
INC = np.radians(72)
T_ORBIT = 2 * np.pi * np.sqrt(R**3 / MU_E)   # orbital period [s]
n = 2 * np.pi / T_ORBIT                      # mean motion [rad/s]
MU_F = 7.95e15                     # Earth's magnetic moment constant [T·m³]

print(f"Orbital period: {T_ORBIT / 60:.1f} min")

def earth_field(t):
    """Earth's magnetic field in the orbit frame [T]: dipole model, paper eq. (1)"""
    return MU_F / R**3 * np.array([
        np.cos(n * t) * np.sin(INC),
        -np.cos(INC),
        2 * np.sin(n * t) * np.sin(INC),
    ])

def skew(w):
    """Cross product as a matrix: skew(a) @ b == np.cross(a, b)"""
    return np.array([[0, -w[2], w[1]], [w[2], 0, -w[0]], [-w[1], w[0], 0]])

def rotate(C, w, dt):
    """Update attitude matrix C after spinning at rate w for dt seconds (Rodrigues' formula)"""
    rate = np.linalg.norm(w)
    if rate < 1e-12:
        return C
    K = skew(w / rate)
    th = rate * dt
    return (np.eye(3) + np.sin(th) * K + (1 - np.cos(th)) * K @ K).T @ C

def simulate(controller, orbits=3, dt=1.0, w0_deg=(1, 3, 1)):
    """Simulate the spacecraft spin with controller(b_now, b_prev, dt) -> dipole moment m.
    Returns (orbits, angular rate [deg/s])."""
    steps = int(orbits * T_ORBIT / dt)
    w = np.radians(w0_deg).astype(float)   # tip-off rate after separation (paper: pitch 3°/s, roll and yaw 1°/s)
    C = np.eye(3)                           # starting attitude
    b_prev = C @ earth_field(0)
    ts, ws = np.zeros(steps), np.zeros((steps, 3))
    for i in range(steps):
        t = i * dt
        b = C @ earth_field(t)              # magnetometer reading (body frame)
        m = controller(b, b_prev, dt)       # dipole moment chosen by the controller
        tau = np.cross(m, b)                # magnetic torque τ = m × B
        w = w + J_INV @ (tau - np.cross(w, J @ w)) * dt   # Euler's rotation equation
        C = rotate(C, w, dt)
        b_prev = b
        ts[i], ws[i] = t, w
    return ts / T_ORBIT, np.degrees(ws)

def no_control(b_now, b_prev, dt):
    return np.zeros(3)

def plot_rates(orbits, rates, title):
    fig, ax = plt.subplots(figsize=(9, 3.5))
    for i, name in enumerate(["roll (x)", "pitch (y)", "yaw (z)"]):
        ax.plot(orbits, rates[:, i], label=name, linewidth=1)
    ax.plot(orbits, np.linalg.norm(rates, axis=1), "k--", label="|ω|", linewidth=1.2)
    ax.set_xlabel("orbits")
    ax.set_ylabel("angular rate [deg/s]")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right")
    plt.show()

def check(ok, hint):
    if ok:
        print("✅ Passed")
    else:
        print("❌ Not yet. " + hint)

K_GAIN = 1e7
print("Ready")

## Problem 1. Build the B-dot controller

$$\dot{\mathbf{B}}_n \approx \frac{\mathbf{B}_n - \mathbf{B}_{n-1}}{\Delta t}, \qquad \mathbf{m} = -k\,\dot{\mathbf{B}}$$

Build it in three steps.
1. `b_dot`: the rate of change of the field
2. `m`: the B-dot law (use `K_GAIN`)
3. Clip to the magnetorquer limit ±`M_MAX` (hint: `np.clip`)

In [ ]:
def my_bdot(b_now, b_prev, dt):
    b_dot = None  # TODO 1
    m = None      # TODO 2
    return m      # TODO 3: clip to the limit

In [ ]:
# Check: does it give the right values for known inputs?
b1 = np.array([2e-5, -1e-5, 3e-5])
t1 = my_bdot(b1, b1 - np.array([1e-6, 2e-6, -1e-6]), 1.0)   # large change → hits the limit
t2 = my_bdot(b1, b1 - np.array([1e-8, -2e-8, 0]), 1.0)     # small change
check(t1 is not None and np.allclose(t1, [-5, -5, 5]), "A large change should be clipped to ±M_MAX.")
check(t2 is not None and np.allclose(t2, [-0.1, 0.2, 0]), "Make sure m = -K_GAIN × b_dot.")

## Problem 2. Stop the satellite with your controller
Run this once the check passes. (It takes a few seconds.)

In [ ]:
if my_bdot(b1, b1, 1.0) is None:
    print("❌ Finish my_bdot in Problem 1 first.")
else:
    orbits, rates = simulate(my_bdot, orbits=3)
    plot_rates(orbits, rates, "My B-dot controller")
    final = np.linalg.norm(rates[-1])
    check(final < 0.2, f"After 3 orbits |ω| = {final:.2f} deg/s. It needs to get below 0.2 deg/s.")

## Challenge. Bang-bang control
On real satellites `k` is hard to tune, so another option is to run the magnetorquers **at full power** all the time and only flip the direction. (Lecture p. 41)

$$m_i = -m^{max}\,\mathrm{sign}(\dot{B}_i)$$

Hint: `np.sign`

In [ ]:
def my_bang_bang(b_now, b_prev, dt):
    return None  # TODO

In [ ]:
# Check
t3 = my_bang_bang(b1, b1 - np.array([1e-8, -2e-8, 0]), 1.0)
check(t3 is not None and np.allclose(t3, [-5, 5, 0]), "Depending on the sign you should get -M_MAX, +M_MAX or 0.")

In [ ]:
if my_bang_bang(b1, b1, 1.0) is None:
    print("❌ Finish my_bang_bang above first.")
else:
    orbits, rates = simulate(my_bang_bang, orbits=3)
    plot_rates(orbits, rates, "Bang-bang B-dot controller")
    print("Compare with the chart from Problem 2. Which one damps faster, and further?")